<a href="https://colab.research.google.com/github/mahmudscode/NLP_work/blob/main/NLP_WORK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Define the target path
target_path = '/content/drive/MyDrive/LLM_BANGLA_DATA/Bangla_newspaper'

# Change the current working directory
os.chdir(target_path)

# Verify the current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/LLM_BANGLA_DATA/Bangla_newspaper


## Step 1: Understand Dataset

To begin understanding the dataset, we first need to load it. Please provide the exact file path and format (e.g., `.csv`, `.json`, directory of `.txt` files) of your '400K Bangla News Articles' dataset. This will allow me to generate the appropriate code for data loading and initial exploration.

First, let's list the contents of the current working directory (`/content/drive/MyDrive/LLM_BANGLA_DATA/Bangla_newspaper`) to identify the dataset files.

In [3]:
import os

# List the contents of the current directory
dir_contents = os.listdir('.')
print("Contents of the current directory:")
for item in dir_contents:
    print(item)

Contents of the current directory:
data
data_v2


Now, let's explore the contents of the `data` and `data_v2` subdirectories to understand the dataset structure and locate the actual news article files.

In [4]:
print("Contents of the 'data' directory:")
data_dir_contents = os.listdir('data')
for item in data_dir_contents:
    print(f"  {item}")

print("\nContents of the 'data_v2' directory:")
data_v2_dir_contents = os.listdir('data_v2')
for item in data_v2_dir_contents:
    print(f"  {item}")

Contents of the 'data' directory:
  data.json

Contents of the 'data_v2' directory:
  data_v2.json


It appears the news articles are stored in `data.json` and `data_v2.json`. Let's load `data.json` into a pandas DataFrame to inspect its structure and content.

In [5]:
import pandas as pd

# Load data.json
try:
    df = pd.read_json('data/data.json')
    print(f"Successfully loaded 'data.json'. Shape: {df.shape}")
    print("First 5 rows of the DataFrame:")
    print(df.head())
    print("\nDataFrame Info:")
    df.info()
except Exception as e:
    print(f"Error loading 'data.json': {e}")

Successfully loaded 'data.json'. Shape: (437948, 10)
First 5 rows of the DataFrame:
              author    category          category_bn        published_date  \
0  গাজীপুর প্রতিনিধি  bangladesh             বাংলাদেশ  ০৪ জুলাই ২০১৩, ২৩:২৬   
1       অনলাইন ডেস্ক      sports                 খেলা  ০৪ জুলাই ২০১৩, ২৩:০৯   
2   নিজস্ব প্রতিবেদক  bangladesh             বাংলাদেশ  ০৪ জুলাই ২০১৩, ২২:২৫   
3       অনলাইন ডেস্ক  technology  বিজ্ঞান ও প্রযুক্তি  ০৪ জুলাই ২০১৩, ২১:৩৭   
4       অনলাইন ডেস্ক  technology  বিজ্ঞান ও প্রযুক্তি  ০৪ জুলাই ২০১৩, ২১:৩৫   

      modification_date          tag comment_count  \
0  ০৪ জুলাই ২০১৩, ২৩:২৭    [গাজীপুর]             0   
1  ০৪ জুলাই ২০১৩, ২৩:১১      [টেনিস]             0   
2  ০৪ জুলাই ২০১৩, ২৩:১২    [রাজনীতি]             0   
3  ০৪ জুলাই ২০১৩, ২১:৪৫     [গবেষণা]             0   
4  ০৪ জুলাই ২০১৩, ২১:৩৭  [কম্পিউটার]             0   

                                               title  \
0  কালিয়াকৈরে টিফিন খেয়ে ৫০০ শ্রমিক অসুস্থ, বিক...   
1   

### Initial Data Exploration: `category_bn` Column

Given that `category_bn` will likely serve as our target variable for classification, let's explore its unique values, their counts, and the number of missing values. This will help us understand the distribution of classes and identify any issues that need addressing during data cleaning.

In [6]:
# Check for missing values in 'category_bn'
missing_categories = df['category_bn'].isnull().sum()
print(f"Number of missing values in 'category_bn': {missing_categories}")

# Display the value counts of 'category_bn'
print("\nValue counts for 'category_bn':")
print(df['category_bn'].value_counts())

Number of missing values in 'category_bn': 123

Value counts for 'category_bn':
category_bn
বাংলাদেশ                                                                                232715
খেলা                                                                                     48369
আন্তর্জাতিক                                                                              30827
বিনোদন                                                                                   30438
অর্থনীতি                                                                                 17229
মতামত                                                                                    15669
বিজ্ঞান ও প্রযুক্তি                                                                      12033
জীবনযাপন                                                                                 10783
শিক্ষা                                                                                    9720
দূর পরবাস                                            

### Handling Missing Values in `category_bn`

Since `category_bn` is our target variable, rows with missing values in this column are not useful for training a classification model. We will drop these rows from the DataFrame.

In [7]:
# Drop rows where 'category_bn' is null
df_cleaned = df.dropna(subset=['category_bn']).copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"DataFrame shape after dropping missing 'category_bn' values: {df_cleaned.shape}")

# Verify no more missing values in 'category_bn'
print(f"Number of missing values in 'category_bn' after cleanup: {df_cleaned['category_bn'].isnull().sum()}")

# Update df to df_cleaned to be used in subsequent steps
df = df_cleaned


Original DataFrame shape: (437948, 10)
DataFrame shape after dropping missing 'category_bn' values: (437825, 10)
Number of missing values in 'category_bn' after cleanup: 0


### Re-evaluating `category_bn` after dropping nulls

Now that the null values have been removed, let's re-examine the `category_bn` value counts to identify and decide on how to handle the low-frequency and irregular categories.

In [8]:
# Display the updated value counts of 'category_bn'
print("\nUpdated value counts for 'category_bn':")
print(df['category_bn'].value_counts())


Updated value counts for 'category_bn':
category_bn
বাংলাদেশ                                                                                232715
খেলা                                                                                     48369
আন্তর্জাতিক                                                                              30827
বিনোদন                                                                                   30438
অর্থনীতি                                                                                 17229
মতামত                                                                                    15669
বিজ্ঞান ও প্রযুক্তি                                                                      12033
জীবনযাপন                                                                                 10783
শিক্ষা                                                                                    9720
দূর পরবাস                                                                                 73

### Further Cleaning of `category_bn`

Based on the `value_counts` output, we observe categories that contain non-Bengali characters and categories with extremely low frequencies. To prepare our target variable for classification, we will:
1.  Remove categories that primarily consist of non-Bengali characters.
2.  Define a frequency threshold and remove categories below this threshold to focus on well-represented classes.

In [ ]:
import re

# Function to check if a string contains mostly Bengali characters
def is_bengali(text):
    # This regex matches any character that is NOT a Bengali character (U+0980 to U+09FF)
    # or common punctuation/digits. We allow some common non-Bengali characters if they are part of a valid name (e.g. '2019' in 'Cricket 2019')
    # However, the goal here is to remove purely garbage entries like 'ржмрж╛ржВрж▓рж╛ржжрзЗрж╢'
    # A simple check for at least one Bengali character and not overwhelmingly non-Bengali might be sufficient for initial filtering.
    bengali_chars = re.findall(r'[\u0980-\u09FF]', str(text))
    total_chars = len(str(text))

    if total_chars == 0: # Handle empty strings
        return False
    # If the proportion of Bengali characters is very low, it's likely noise
    return len(bengali_chars) / total_chars > 0.5 # A threshold to consider it mostly Bengali


# Filter out categories that are not mostly Bengali
df_filtered_bengali = df[df['category_bn'].apply(is_bengali)].copy()

print(f"DataFrame shape after filtering non-Bengali categories: {df_filtered_bengali.shape}")

# Set a frequency threshold for categories
# Let's consider categories with less than 100 articles as 'low-frequency'
frequency_threshold = 100

# Get value counts of remaining categories
category_counts = df_filtered_bengali['category_bn'].value_counts()

# Identify categories to keep (above the threshold)
categories_to_keep = category_counts[category_counts >= frequency_threshold].index

# Filter the DataFrame to keep only articles with these categories
df_final_categories = df_filtered_bengali[df_filtered_bengali['category_bn'].isin(categories_to_keep)].copy()

print(f"DataFrame shape after filtering low-frequency categories (threshold={frequency_threshold}): {df_final_categories.shape}")

# Update df to the cleaned DataFrame
df = df_final_categories

# Display the final value counts and number of unique categories
print("\nFinal value counts for 'category_bn' after cleaning:")
print(df['category_bn'].value_counts())
print(f"\nNumber of unique categories after cleaning: {df['category_bn'].nunique()}")